In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('DATA_CUT.csv')

#Histograms & KDE
num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(20, 16))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='#4C72B0', bins=25, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig('hist_kde_plot_datacut.png', dpi=300)
print("![Histograms & KDE](hist_kde_plot_datacut.png)")
plt.close()

print("\n=== DATATYPES ===")
print(df.dtypes.to_string())

print("\n=== MISSING VALUES ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
miss_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
miss_df = miss_df[miss_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if not miss_df.empty:
    print(miss_df.to_string())
print(f"\nTổng ô thiếu: {missing.sum()}")
print(f"Features bị thiếu: {(missing > 0).sum()}/{len(df.columns)}")

if not miss_df.empty:
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=miss_df.index, y='Missing %', data=miss_df, palette='flare')
    for p in ax.patches:
        ax.annotate(f'{p.get_height()}%',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom',
                    fontsize=10, fontweight='bold', color='black', xytext=(0, 5), textcoords='offset points')
    plt.title('Percentage of Missing Data by Feature', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Missing Percentage (%)', fontsize=12)
    plt.xlabel('Features', fontsize=12)
    plt.ylim(0, max(miss_df['Missing %']) + 5) 
    plt.tight_layout()
    plt.savefig('missing_data_plot_datacut.png', dpi=300)
    print("\n![Missing Data Plot](missing_data_plot_datacut.png)")
    plt.close()
else:
    print("\nTuyệt vời! Tập dữ liệu không có giá trị bị thiếu (No missing values).")

# 5. Outlier Detection (IQR Method)
print("\n=== OUTLIER DETECTION (IQR Method) ===")
outlier_rows = set()
for col in num_cols:
    s = df[col].dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lb, ub = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    mask = (df[col] < lb) | (df[col] > ub)
    n_out = mask.sum()
    outlier_rows.update(df[mask].index.tolist())
    print(f"  {col:<28} | Outliers: {n_out} ({n_out/len(s)*100:.2f}%) | Range: [{lb:.1f}, {ub:.1f}]")

print(f"\nTổng rows có ít nhất 1 outlier: {len(outlier_rows)}")